# DenseUAV: tối ưu + CT1–CT3 trên baseline Global (CE + HardMining Triplet), Kaggle GPU T4 ×2

Giống `denseuav_optimize_t4x2.ipynb` (cùng code, cùng các bước), chỉ khác baseline: **head Global (class token) với cấu hình Bảng VI của bài báo**. Cụ thể: ViT-S/224, CE + HardMiningTripletLoss, **không KL**, SGD lr 0.003, bs 16, 120 epoch, LR ×0.1 ở 70/110, augment `rr=uav`, `ra/re=satellite`. Bài báo đạt 80.18 R@1 / 84.39 SDM@1; lần tái hiện trước (`baselinetest03`, full-res) đạt 79.45 / 83.92. Mỗi run chỉ đổi **một** thứ so với `global`; riêng `vitb_global_dss_lc` ghép các thứ lại.

| # | Tối ưu | Bằng chứng | Cách áp dụng ở đây | Run |
|---|---|---|---|---|
| O1 | Giải mã ảnh drone một lần (cache cạnh ngắn 512 px) + 2 run song song, mỗi T4 một run | Run cũ ~1 h 50, nhiều khả năng nghẽn CPU; `train.log` in thời gian mỗi epoch | `build_cache`, hàng đợi 2 GPU | tất cả |
| O2 | Backbone **ViT-B/16** | Bảng V của DenseUAV: ViT-B 87.82 vs ViT-S 80.18 R@1, đúng head Global | `--backbone ViTB-224` | `vitb_global` |
| O3 | Backbone tự giám sát **DINOv2 ViT-S/14** | Yang et al. (RA-L 2025) báo cáo DINOv2-B 86.27 R@1 / 88.87 SDM@1 | `DINOv2S-224` | `dino_global` |
| O4 | **Lấy mẫu động** (láng giềng GPS + lớp dễ nhầm trong batch) | CEUSP (arXiv 2502.11408), Bảng V–VII: ngẫu nhiên 85.84 → trộn 89.45 R@1; chỉ GPS sụp còn 52.72 | `DSSampler` | `global_dss` |
| CT2 | **Nhất quán không gian cục bộ mềm** (loss thứ tự) | Ý tưởng 2. Lần trước +1.11 R@1 trên Global, nhưng khi đó dùng cấu hình repo (WeightedSoftTriplet + KL) | `local_consistency_loss` | `global_ct2`, `global_dss_lc`, `vitb_global_dss_lc` |
| CT1 | **Head IRP** (tích phân Lebesgue ring × sector, bất biến hướng bay): class token + 2 nhánh vòng | Ý tưởng 1 + Ví dụ 4.1 của bài Lebesgue | `head=IRP`, `block=2`, **lr 0.01** (bài báo dùng 0.01 cho mọi head chia khối, Mục VI-E), loss giữ nguyên | `irp` |
| O5 | Test-time: TTA xoay D4 + DBA 2 năm | Kỹ thuật retrieval chuẩn; chưa ai báo cáo trên DenseUAV | phần phân tích | tất cả |
| CT3 | **Chẩn đoán lỗi** C0–C4, ma trận chuyển so với `global` | Ý tưởng 3 | phần phân tích | tất cả |

Head Global là cấu hình yếu hơn LPN, nên các cải tiến có nhiều chỗ để thấy tác dụng. So với `denseuav_optimize_t4x2.ipynb`, có thể xem cải tiến nào vẫn giữ được khi baseline đã mạnh.

**Settings:** Accelerator `GPU T4 x2`, Internet `On`, Input `netvoych1k/denseuav`, *Save & Run All*. Run nào không kịp xong trước 10 h 45 sẽ bị bỏ qua; phần phân tích vẫn chạy.

**Đọc kết quả:** 1 seed mỗi cấu hình, riêng `global` có thêm seed 777 để đo nhiễu (lần trước khoảng ±1 R@1). Cấu hình test-time `D4 both + DBA years` được chọn trước khi chạy.


In [1]:
import glob, json, os, re, shutil, subprocess, sys, time
ROOT = os.environ.get('DENSEUAV_ROOT') or os.path.dirname(glob.glob('/kaggle/input/**/Dense_GPS_ALL.txt', recursive=True)[0])
WORK = os.environ.get('WORK', '/kaggle/working')
REPO, DATA, LOGS = f'{WORK}/DenseUAV', f'{WORK}/cache', f'{WORK}/logs'  # DATA = ROOT with the drone images shrunk once (O1)
CK = f'{REPO}/checkpoints'
NW = os.cpu_count()  # DataLoader workers in the analysis
if 'DENSEUAV_ROOT' not in os.environ:  # class counts from the repo README
    for d, n in [('train/drone', 2256), ('train/satellite', 2256), ('test/query_drone', 777), ('test/gallery_satellite', 3033)]:
        assert len(os.listdir(f'{ROOT}/{d}')) == n, d
print(ROOT, '|', os.cpu_count(), 'vCPUs')
subprocess.run(['nvidia-smi', '-L'])

/kaggle/input/datasets/netvoych1k/denseuav/DenseUAV | 4 vCPUs
GPU 0: Tesla T4 (UUID: GPU-f8d3b1db-dddb-43eb-aed5-aa0d8c172db6)
GPU 1: Tesla T4 (UUID: GPU-0a605808-e9ff-27e4-29c0-33f82d069ed7)


CompletedProcess(args=['nvidia-smi', '-L'], returncode=0)

In [2]:
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/Dmmm1997/DenseUAV.git', REPO], check=True)
    subprocess.run(['git', '-C', REPO, 'checkout', '-q', 'b8de187'], check=True)
os.chdir(REPO)
sys.path.insert(0, REPO)
for p in ('datasets', 'losses', 'optimizers', 'tool'):
    open(f'{p}/__init__.py', 'a').close()

In [3]:
%pip install -q -r /kaggle/working/DenseUAV/requirments.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.8/127.8 kB 3.8 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


## Code mới (3 file) và patch repo
- `geo_utils.py`: GPS → mét, `GeoSampler` (CT2 cũ), `mixed_order` (thứ tự nhóm của O4), `local_consistency_loss` (CT2), metric R@K/SDM@K viết lại theo đúng định nghĩa của repo, và các loại lỗi của CT3.
- `models/Head/IRP.py` (CT1): với U_kj = vòng k × cung j của đĩa nội tiếp ảnh (tâm ảnh = vị trí UAV, vì camera nhìn thẳng xuống) và f(y) = max(0, 1 − |y|/R), mỗi ô cho
  W[k, j] = ∫_{U_kj} f(y) φ(y) dy / ∫_{U_kj} f(y) dy, với φ = patch token của ViT.
  Tích phân được tính bằng tổng Riemann trên lưới con 8×8 mỗi patch. Xoay ảnh làm các cung dịch vòng, lật ảnh làm thứ tự cung đảo ngược. Vì vậy mỗi vòng chỉ giữ hai thành phần không đổi theo hướng bay: trung bình theo cung (DFT bậc 0, giữ dấu) và |DFT bậc 1|. Head có cùng dạng đầu ra với LPN (class token + 1 nhánh mỗi vòng), nên loss của repo và `test.py` dùng được ngay.
- `train_geo.py` (O4, CT2): `train.py` của repo, chỉ thay sampler và cộng thêm loss.
  - `GEO_SAMPLER=geo|dss` chọn sampler, `LC=` là trọng số loss.
  - Từ epoch 20, `DSSampler` đo độ tương đồng drone → vệ tinh của 2256 lớp train (1 ảnh mỗi view, transform của `test.py`) mỗi 6 epoch, và lấy 16 lớp gần nhất làm lớp dễ nhầm.

Các patch còn lại trên repo:
- Tương thích numpy 2 / timm 1.x (giống các notebook trước).
- Seed lấy từ biến môi trường.
- Thêm backbone `DINOv2S-224`, đăng ký head `IRP`.

Self-test chạy vài giây, trước khi bắt đầu train.

In [4]:
%%writefile geo_utils.py
"""Helpers for CT2 (neighbour-aware sampling, local spatial consistency) and CT3 (error diagnosis)."""
import math
import numpy as np
import torch
import torch.nn.functional as F
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from scipy.spatial import cKDTree

EARTH_R = 6378137.0  # same radius as evaluateDistance.latlog2meter


def load_gps(root):
    """class folder name -> (lon, lat), parsed the same way as evaluateDistance.py."""
    gps = {}
    for line in open(f'{root}/Dense_GPS_ALL.txt'):
        p = line.split(' ')
        gps[p[0].split('/')[-2]] = (float(p[1].split('E')[-1]), float(p[2].split('N')[-1]))
    return gps


def class_xy(names, gps):
    """Class positions in metres around the mean of all GPS points (equirectangular projection:
    about 0.5% scale error 50 km from the reference, negligible for the 20-60 m thresholds used here)."""
    lon0, lat0 = np.mean(list(gps.values()), 0)
    ll = np.array([gps[n] for n in names], dtype=np.float64)
    x = np.radians(ll[:, 0] - lon0) * math.cos(math.radians(lat0)) * EARTH_R
    y = np.radians(ll[:, 1] - lat0) * EARTH_R
    return np.stack([x, y], 1)


# ---------------- CT2: neighbour-aware sampling and local spatial consistency ----------------

class GeoSampler(torch.utils.data.Sampler):
    """Random permutation of the classes in which every class is followed by up to group-1 unused
    neighbours within `radius` m. Each class still appears once per epoch (as Sampler_University with
    sample_num=1), so labels inside a batch stay distinct, which the repo's triplet mining needs."""

    def __init__(self, xy, group=4, radius=30.0):
        tree = cKDTree(xy)
        self.nbrs = [[j for j in tree.query_ball_point(p, radius) if j != i] for i, p in enumerate(xy)]
        self.group = group

    def __iter__(self):
        used = np.zeros(len(self.nbrs), bool)
        order = []
        for a in np.random.permutation(len(self.nbrs)):
            if used[a]:
                continue
            free = [j for j in self.nbrs[a] if not used[j]]
            np.random.shuffle(free)
            g = [a] + free[:self.group - 1]
            used[g] = True
            order += g
        return iter(order)

    def __len__(self):
        return len(self.nbrs)


def mixed_order(nbrs, feat_nn, n_geo=1, n_feat=2):
    """One epoch of class indices, every class once (CEUSP's dynamic sampling, scaled to batch 16):
    groups of a random anchor + up to n_geo unused GPS neighbours + up to n_feat unused classes that the model
    currently confuses with the anchor (feat_nn[a], most similar first). The next anchor is random again."""
    used = np.zeros(len(nbrs), bool)
    order = []
    for a in np.random.permutation(len(nbrs)):
        if used[a]:
            continue
        used[a] = True
        geo = [j for j in np.random.permutation(nbrs[a]) if not used[j]][:n_geo]
        used[geo] = True
        hard = [j for j in feat_nn[a] if not used[j]][:n_feat]
        used[hard] = True
        order += [a] + geo + hard
    return order


def local_consistency_loss(feat, xy, near=30.0, far=60.0, alpha=3.0):
    """Ordinal local consistency: for every sample that has a <= `near` m neighbour in the batch, its
    hardest neighbour must be closer in feature space than its closest place > `far` m away.
    No Lipschitz constant is imposed, only the order near < far. Same soft form and alpha as the
    repo's WeightedSoftTripletLoss. feat: (n, d) features of both views, xy: (n, 2) metres."""
    f = F.normalize(feat.float(), dim=1)
    d_f = (2 - 2 * f @ f.T).clamp_min(1e-12).sqrt()
    d_g = torch.cdist(xy, xy)
    is_near = (d_g > 0) & (d_g <= near)
    is_far = d_g > far
    ok = is_near.any(1) & is_far.any(1)
    if not ok.any():
        return d_f.new_zeros(())
    d_near = d_f.masked_fill(~is_near, float('-inf')).amax(1)[ok]
    d_far = d_f.masked_fill(~is_far, float('inf')).amin(1)[ok]
    return F.softplus(alpha * (d_near - d_far)).mean()


# ---------------- metrics (same definitions as evaluate_gpu.py / evaluateDistance.py) ----------------

def recall(top, q_lab, g_lab, k):
    return 100 * (g_lab[top[:, :k]] == q_lab[:, None]).any(1).mean()


def sdm(top, q_ll, g_ll, k):
    """SDM@K as evaluateDistance.py: Euclidean distance in degrees, exp(-5e3 d), weights (K - i) / K."""
    d = np.linalg.norm(g_ll[top[:, :k]] - q_ll[:, None], axis=-1)
    w = (k - np.arange(k)) / k
    return 100 * ((np.exp(-5e3 * d) * w).sum(1) / w.sum()).mean()


# ---------------- CT3: error diagnosis ----------------

def areas(xy, gap=200.0):
    """Connected components of the sampling points with links <= gap m (one per flight area / campus)."""
    pairs = cKDTree(xy).query_pairs(gap, output_type='ndarray')
    adj = coo_matrix((np.ones(len(pairs)), pairs.T), shape=(len(xy), len(xy)))
    return connected_components(adj, directed=False)[1]


def error_types(pred, gt, xy, area, is_test, near=30.0):
    """Per query: C0 correct, C1 wrong cell <= near m away, C2 same area > near m, C3 another test area,
    C4 a training location (train and test universities are disjoint, so C4 is always inter-campus)."""
    d = np.linalg.norm(xy[pred] - xy[gt], axis=1)
    cat = np.select([pred == gt, d <= near, ~is_test[pred], area[pred] == area[gt]],
                    ['C0', 'C1', 'C4', 'C2'], 'C3')
    return cat, d


def _selftest():
    rng = np.random.default_rng(0)
    # 20 m grid of two 10x10 areas 5 km apart
    g = np.stack(np.meshgrid(np.arange(10), np.arange(10)), -1).reshape(-1, 2) * 20.0
    xy = np.concatenate([g, g + 5000])
    n = len(xy)
    np.random.seed(0)

    def pairs_per_batch(o):
        b = xy[np.array(o[:len(o) // 16 * 16])].reshape(-1, 16, 2)
        d = np.linalg.norm(b[:, :, None] - b[:, None], axis=-1)
        return ((d > 0) & (d <= 30)).sum() / 2 / len(b)

    # GeoSampler: a permutation, neighbours mostly together
    sampler = GeoSampler(xy)
    order = list(sampler)
    assert sorted(order) == list(range(n))
    rnd = list(np.random.permutation(n))
    assert pairs_per_batch(order) > 3 * pairs_per_batch(rnd)
    # mixed_order: a permutation, and each class meets its top confuser in the same batch far more often than at random
    feat_nn = np.array([rng.permutation(np.delete(np.arange(n), i))[:16] for i in range(n)])
    order = mixed_order(sampler.nbrs, feat_nn)
    assert sorted(order) == list(range(n))
    batch = np.empty(n, int)
    batch[np.array(order)] = np.arange(n) // 16
    together = lambda b: (b == b[feat_nn[:, 0]]).mean()
    batch_rnd = np.empty(n, int)
    batch_rnd[np.array(rnd)] = np.arange(n) // 16
    assert together(batch) > 3 * together(batch_rnd), (together(batch), together(batch_rnd))
    assert pairs_per_batch(order) > 1.3 * pairs_per_batch(rnd)  # one GPS neighbour per group of 4 (toy grid is dense)
    # loss: ~0 when neighbours are close and far places far, large when reversed
    p = torch.tensor(xy[[0, 1, 150]], dtype=torch.float32)  # 0-1 are 20 m apart, 150 is 5 km away
    good = torch.tensor([[1., 0], [0.9, 0.1], [0, 1]])
    bad = torch.tensor([[1., 0], [0, 1], [0.9, 0.1]])
    assert local_consistency_loss(good, p) < 0.2 < 1.0 < local_consistency_loss(bad, p)
    # metrics on a toy ranking
    qf = F.normalize(torch.randn(5, 8), dim=1).numpy()
    gf = F.normalize(torch.randn(30, 8), dim=1).numpy()
    top = np.argsort(-(qf @ gf.T), 1)
    lab = np.arange(30)
    assert recall(top, top[:, 0], lab, 1) == 100 and recall(top, top[:, 3], lab, 3) == 0
    ll = rng.random((30, 2))
    assert abs(sdm(top, ll[top[:, 0]], ll, 1) - 100) < 1e-9
    # diagnosis categories
    area = areas(xy)
    assert area.max() == 1
    is_test = np.arange(n) < 100  # first area = test, second = train
    cat, _ = error_types(np.array([0, 1, 50, 150]), np.array([0, 0, 0, 0]), xy, area, is_test)
    assert list(cat) == ['C0', 'C1', 'C2', 'C4'], cat
    print('geo_utils self-test passed')


if __name__ == '__main__':
    _selftest()

Writing geo_utils.py


In [5]:
%%writefile models/Head/IRP.py
"""Integral ring pooling (IRP): CT1 inside the network, an LPN-style head built from the Lebesgue shape vector.
Component (k, j) is the cell average of the patch tokens over U_kj = ring k x sector j of the disc inscribed
in the image, weighted by f(y) = max(0, 1 - |y|/R) (f_decay, Example 4.1 of the Lebesgue paper). The image
centre is the UAV position (nadir camera). A rotation of the image shifts the sectors cyclically and a flip
reverses them, so each ring keeps the sector mean (DFT harmonic 0, signed) and |harmonic 1|: both ignore the
unknown heading. Output format = LPN (global class token + one branch per ring), so losses and test.py work as is."""
import math
import torch
import torch.nn as nn
from .utils import ClassBlock

N_SECTORS = 8


def ring_sector_kernels(n_r, n_t, h, sub=8):
    """(n_r, n_t, h*h): K[k, j, p] = integral of f over U_kj inside patch p, divided by the integral of f over U_kj.
    Riemann sum on an (h*sub)^2 grid of the image square [-1, 1]^2, R = 1."""
    s = h * sub
    u = (torch.arange(s) + 0.5) / s * 2 - 1
    y, x = torch.meshgrid(u, u, indexing='ij')
    r = (x ** 2 + y ** 2).sqrt()
    ring = (r * n_r).long().clamp_max(n_r - 1)
    # sectors start half a sector off the axes: no pixel centre lies on a boundary, so the grid partition is
    # exactly symmetric under 90 degree rotations and flips
    t = (torch.atan2(y, x) + math.pi / n_t) % (2 * math.pi)
    sec = (t / (2 * math.pi) * n_t).long().clamp_max(n_t - 1)
    k = torch.zeros(n_r, n_t, s, s)
    k[ring, sec, torch.arange(s)[:, None], torch.arange(s)[None]] = (1 - r).clamp_min(0)  # 0 outside the disc
    k = k.reshape(n_r, n_t, h, sub, h, sub).sum((3, 5)).flatten(2)
    return k / k.sum(-1, keepdim=True)


class IRP(nn.Module):
    def __init__(self, opt):
        super().__init__()
        self.block = opt.block  # number of rings, as LPN's block
        self.global_classifier = ClassBlock(opt.in_planes, opt.nclasses, opt.droprate, num_bottleneck=opt.num_bottleneck)
        self.ring_classifiers = nn.ModuleList(
            ClassBlock(2 * opt.in_planes, opt.nclasses, opt.droprate, num_bottleneck=opt.num_bottleneck)
            for _ in range(opt.block))
        self.kernel = None  # built on the first call, when the patch grid size is known

    def ring_features(self, tokens):
        """tokens (B, h*h, C) -> (B, rings, 2C) = [sector mean, |DFT harmonic 1| over the sectors]."""
        h = int(round(tokens.shape[1] ** 0.5))
        if self.kernel is None or self.kernel.shape[-1] != h * h or self.kernel.device != tokens.device:
            self.kernel = ring_sector_kernels(self.block, N_SECTORS, h).to(tokens.device)
        with torch.autocast(tokens.device.type, enabled=False):
            w = torch.einsum('bpc,ktp->bktc', tokens.float(), self.kernel)
            z = torch.fft.rfft(w, dim=2)
            mag1 = (z[:, :, 1].real ** 2 + z[:, :, 1].imag ** 2 + 1e-8).sqrt()
            return torch.cat([z[:, :, 0].real / N_SECTORS, mag1 / N_SECTORS], -1)

    def forward(self, features):
        rings = self.ring_features(features[:, 1:])
        outs = [self.global_classifier(features[:, 0])] + [c(rings[:, i]) for i, c in enumerate(self.ring_classifiers)]
        cls, feats = [o[0] for o in outs], [o[1] for o in outs]
        if not self.training:
            feats = torch.stack(feats, dim=-1)
        return [cls, feats]

Writing models/Head/IRP.py


In [6]:
%%writefile train_geo.py
# CT2 / O4: train.py with a neighbour-aware sampler and the local-consistency loss. Only the sampler and the
# loss change; the parser, model, optimizer and training loop are train.py's own, so runs stay comparable.
#   GEO_SAMPLER=geo  GeoSampler: each class followed by up to 3 GPS neighbours <= 30 m (the earlier CT2 run)
#   GEO_SAMPLER=dss  CEUSP dynamic sampling: random until epoch 20, then anchor + 1 GPS neighbour + 2 confusers
#   LC=<weight>      local-consistency loss weight (0 = off)
import os
import numpy as np
import torch
import torch.nn.functional as F
import torch.backends.cudnn as cudnn
from PIL import Image
from torchvision import transforms
import train as T
from datasets.make_dataloader import make_dataset
from datasets.Dataloader_University import train_collate_fn
from losses.loss import Loss
from models.taskflow import make_model
from optimizers.make_optimizer import make_optimizer
from tool.utils import copyfiles2checkpoints, set_seed
from geo_utils import load_gps, class_xy, GeoSampler, mixed_order, local_consistency_loss

SAMPLER = os.environ.get('GEO_SAMPLER', 'geo')
LC_WEIGHT = float(os.environ.get('LC', '1'))
DSS_START, DSS_EVERY = 20, 6  # CEUSP Sec. III-D: dynamic sampling from epoch 20, similarities refreshed every 6 epochs


class GeoLoss(Loss):
    def __init__(self, opt, xy):
        super().__init__(opt)
        self.xy = xy

    def forward(self, outputs, outputs2, labels, labels2):
        loss, cls_loss, triplet_loss, kl_loss = super().forward(outputs, outputs2, labels, labels2)
        if LC_WEIGHT == 0:
            return loss, cls_loss, triplet_loss, kl_loss
        f1, f2 = outputs[1], outputs2[1]
        parts = list(zip(f1, f2)) if isinstance(f1, list) else [(f1, f2)]  # LPN/IRP: one term per part, as calc_triplet_loss
        xy = self.xy[torch.cat((labels, labels2))]
        lc = LC_WEIGHT * sum(local_consistency_loss(torch.cat(p), xy) for p in parts) / len(parts)
        # train.py logs four fixed numbers, so the LC term is reported inside "Triplet_Loss"
        return loss + lc, cls_loss, triplet_loss + lc, kl_loss


class Files(torch.utils.data.Dataset):
    def __init__(self, files, tf):
        self.files, self.tf = files, tf

    def __len__(self):
        return len(self.files)

    def __getitem__(self, i):
        return self.tf(Image.open(self.files[i]).convert('RGB'))


class DSSampler(torch.utils.data.Sampler):
    """Every class once per epoch. Epochs < DSS_START: random order (as Sampler_University). Later: mixed_order groups,
    where the confusers are the classes whose satellite view is most similar to the anchor's drone view under the
    current model (one fixed image per class and view, test.py's transform and part normalisation)."""

    def __init__(self, xy, ds, opt):
        self.nbrs = GeoSampler(xy).nbrs
        self.files = [sorted(ds.dict_path[v][c])[0] for v in ('drone', 'satellite') for c in ds.cls_names]
        self.tf = transforms.Compose([transforms.Resize((opt.h, opt.w), interpolation=3), transforms.ToTensor(),
                                      transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
        self.workers, self.epoch, self.feat_nn, self.model = opt.num_worker, 0, None, None

    def __len__(self):
        return len(self.nbrs)

    @torch.no_grad()
    def confusers(self, k=16):
        self.model.eval()
        out = []
        for x in torch.utils.data.DataLoader(Files(self.files, self.tf), 128, num_workers=self.workers):
            with torch.autocast('cuda'):
                f = self.model(x.cuda(), None)[0][1].float()
            out.append(F.normalize(F.normalize(f, dim=1).flatten(1), dim=1))
        self.model.train()
        f = torch.cat(out)
        n = len(self.nbrs)
        s = f[:n] @ f[n:].T  # drone (rows) -> satellite (columns)
        s.fill_diagonal_(-2)
        return s.topk(k, dim=1).indices.cpu().numpy()

    def __iter__(self):
        e, self.epoch = self.epoch, self.epoch + 1  # the DataLoader asks for a new iterator once per epoch
        if e < DSS_START:
            return iter(np.random.permutation(len(self.nbrs)).tolist())
        if (e - DSS_START) % DSS_EVERY == 0:
            self.feat_nn = self.confusers()
        return iter(mixed_order(self.nbrs, self.feat_nn))


if __name__ == '__main__':
    set_seed(int(os.environ.get('SEED', 666)))
    opt = T.get_parse()
    opt.use_gpu = torch.cuda.is_available()
    torch.cuda.set_device(int(opt.gpu_ids.split(',')[0]))
    cudnn.benchmark = True

    loader, class_names, dataset_sizes = make_dataset(opt)
    xy = class_xy(class_names, load_gps(os.path.dirname(opt.data_dir.rstrip('/'))))
    sampler = GeoSampler(xy) if SAMPLER == 'geo' else DSSampler(xy, loader.dataset, opt)
    loader = torch.utils.data.DataLoader(loader.dataset, batch_size=opt.batchsize, sampler=sampler,
                                         num_workers=opt.num_worker, pin_memory=True, collate_fn=train_collate_fn)
    opt.nclasses = len(class_names)
    model = make_model(opt)
    optimizer, scheduler = make_optimizer(model, opt)
    model = model.cuda()
    sampler.model = model
    copyfiles2checkpoints(opt)
    xy_t = torch.tensor(xy, dtype=torch.float32).cuda()
    T.Loss = lambda o: GeoLoss(o, xy_t)  # train_model builds its loss with Loss(opt)
    T.train_model(model, opt, optimizer, scheduler, loader, dataset_sizes)

Writing train_geo.py


In [7]:
def patch(path, old, new):
    """Replace every `old` by `new` in a repo file; running the cell again is a no-op."""
    s = open(path, encoding='utf-8').read()
    if new not in s:
        assert old in s, (path, old)
        open(path, 'w', encoding='utf-8').write(s.replace(old, new))


# numpy 2 / timm 1.x, same fixes as the earlier notebooks
patch('evaluateDistance.py', 'np.bool8', 'np.bool_')
for f in ('evaluateDistance.py', 'evaluate_gpu.py'):
    patch(f, 'np.in1d', 'np.isin')
patch('models/Backbone/cvt.py', 'timm.models.layers', 'timm.layers')
patch('tool/get_inference_time.py', 'yaml.load(stream)', 'yaml.safe_load(stream)')
patch('tool/get_inference_time.py', 'print("inference_time = {}s".format(time.time()-since))',
      'print(f"Params={params} Macs={macs} InferTime={time.time()-since:.1f}s")')
# seed from the environment (global_seed777 measures run-to-run noise)
patch('train.py', '    set_seed(666)', "    set_seed(int(__import__('os').environ.get('SEED', 666)))")
# O3: DINOv2 ViT-S/14 (LVD-142M); at 224 px it has 16 x 16 patches, timm resamples the position embedding
patch('models/Backbone/backbone.py', '        elif backbone=="ViTS-384":',
      '        elif backbone=="DINOv2S-224":\n'
      '            backbone_model = timm.create_model("vit_small_patch14_dinov2.lvd142m", pretrained=True, img_size=self.img_size)\n'
      '            output_channel = 384\n'
      '        elif backbone=="ViTS-384":')
# CT1: register the IRP head
patch('models/Head/head.py', 'from .LPN import LPN, LPN_CNN', 'from .LPN import LPN, LPN_CNN\nfrom .IRP import IRP')
patch('models/Head/head.py', '        elif head == "GeM":', '        elif head == "IRP":\n            head_model = IRP(opt)\n        elif head == "GeM":')

# self-tests
import timm, torch
from types import SimpleNamespace
import geo_utils
from models.Head.IRP import IRP
geo_utils._selftest()
irp = IRP(SimpleNamespace(block=2, in_planes=16, nclasses=10, droprate=0.5, num_bottleneck=32))
for h in (14, 16):  # ViT-S/16 and DINOv2-S/14 at 224 px
    grid = torch.randn(2, h, h, 16)
    r0 = irp.ring_features(grid.flatten(1, 2))
    for g in (grid.rot90(1, (1, 2)), grid.rot90(2, (1, 2)), grid.flip(2)):  # heading changes
        assert (irp.ring_features(g.flatten(1, 2)) - r0).abs().max() < 1e-4 * r0.abs().max()
    assert (irp.ring_features(grid.flatten(1, 2)[:, torch.randperm(h * h)]) - r0).abs().mean() > 0.1 * r0.abs().mean()
dino = timm.create_model('vit_small_patch14_dinov2', pretrained=False, img_size=(224, 224))
assert dino.forward_features(torch.zeros(1, 3, 224, 224)).shape == (1, 257, 384)  # class token + 16 x 16, no registers
subprocess.run([sys.executable, '-c', 'import train_geo'], check=True)  # import errors now, not after hours of training
print('IRP self-test passed')

geo_utils self-test passed


/kaggle/working/DenseUAV/models/Backbone/cvt.py:553: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
  or pretrained_layers[0] is '*'


IRP self-test passed


## O1: cache ảnh drone
Ảnh drone gốc là JPEG 1440×1080, mỗi epoch được giải mã lại ở độ phân giải đầy đủ chỉ để rồi resize về 224. Cell này giải mã một lần ở 1/2 kích thước (JPEG draft), resize về cạnh ngắn 512 bằng bicubic rồi lưu JPEG q95, giữ nguyên tên file.
- Tỉ lệ khung không đổi, nên mọi transform của repo vẫn chạy y như cũ: `RotateAndCrop` lấy hình vuông nội tiếp đường tròn (362 px ≥ 224), sau đó `Resize((224, 224))`.
- Ảnh vệ tinh (512 px) và `Dense_GPS_ALL.txt` được link nguyên trạng.
- Baseline `global` cũng chạy trên cache này, nên mọi so sánh trong notebook đều công bằng.

In [8]:
from multiprocessing.pool import ThreadPool
from PIL import Image


def link(src, dst):
    if not os.path.lexists(dst):
        try:
            os.symlink(src, dst)
        except OSError:  # no symlink rights (Windows): copy instead
            (shutil.copytree if os.path.isdir(src) else shutil.copy)(src, dst)


def shrink(job, short=512):
    src, dst = job
    if os.path.exists(dst):
        return
    im = Image.open(src)
    im.draft('RGB', (im.width * short // min(im.size), im.height * short // min(im.size)))  # JPEG: decode at 1/2 scale
    im = im.convert('RGB')
    r = short / min(im.size)
    if r < 1:
        im = im.resize((round(im.width * r), round(im.height * r)), Image.BICUBIC)
    im.save(dst, 'JPEG', quality=95)


t = time.time()
for split in ('train', 'test'):
    os.makedirs(f'{DATA}/{split}', exist_ok=True)
link(f'{ROOT}/Dense_GPS_ALL.txt', f'{DATA}/Dense_GPS_ALL.txt')
link(f'{ROOT}/train/satellite', f'{DATA}/train/satellite')
link(f'{ROOT}/test/gallery_satellite', f'{DATA}/test/gallery_satellite')
jobs = []
for view in ('train/drone', 'test/query_drone'):
    for c in sorted(os.listdir(f'{ROOT}/{view}')):
        os.makedirs(f'{DATA}/{view}/{c}', exist_ok=True)
        jobs += [(f'{ROOT}/{view}/{c}/{f}', f'{DATA}/{view}/{c}/{f}') for f in os.listdir(f'{ROOT}/{view}/{c}')
                 if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
with ThreadPool(os.cpu_count()) as pool:
    pool.map(shrink, jobs, chunksize=32)
print(f'{len(jobs)} drone images: {Image.open(jobs[0][0]).size} -> {Image.open(jobs[0][1]).size}, '
      f'{sum(os.path.getsize(d) for _, d in jobs) / 2**30:.2f} GB, {time.time() - t:.0f} s')

9099 drone images: (1440, 1080) -> (683, 512), 1.37 GB, 100 s


## Các run
`BASE` = `train_test_local.sh` của repo với cấu hình Global của Bảng VI (`head_pool=global`, HardMining, không KL, lr 0.003). Mỗi run gồm 4 bước như script gốc, thêm một bước đo Params/MACs/InferTime:
`train → test.py → evaluate_gpu.py → evaluateDistance.py → tool/get_inference_time.py`

**Cách chạy:**
- Hai luồng, mỗi luồng giữ một GPU (`CUDA_VISIBLE_DEVICES`) và lần lượt lấy run tiếp theo trong danh sách ưu tiên.
- Trước khi bắt đầu một run, luồng ước lượng thời gian của nó: thời gian thật của các run đã xong × hệ số backbone. Nếu run sẽ kết thúc sau `DEADLINE_H`, nó bị bỏ qua.
- Log của mỗi run nằm ở `/kaggle/working/logs/<run>.log`; thời gian mỗi epoch nằm ở `checkpoints/<run>/train.log`.
- Run nào đã có kết quả (chạy lại cell) sẽ không bị train lại.

InferTime đo trong lúc GPU kia vẫn đang train, nên chỉ dùng để so giữa các run với nhau.

In [9]:
import queue
import numpy as np
import torch
from concurrent.futures import ThreadPoolExecutor, as_completed

T0 = time.time()
DEADLINE_H = 10.75  # no run may end later: Kaggle stops at 12 h and the analysis needs ~45 min
COST = {'ViTS-224': 1.0, 'DINOv2S-224': 1.4, 'ViTB-224': 3.5}  # training time relative to ViT-S (rough, from MACs)
BASE = dict(gpu_ids='0', num_worker=max(2, os.cpu_count() // 2), lr=0.003, batchsize=16, sample_num=1, block=1,
            num_bottleneck=512, backbone='ViTS-224', head='SingleBranch', head_pool='global', cls_loss='CELoss',
            feature_loss='HardMiningTripletLoss', kl_loss='no', h=224, w=224, load_from='no',
            ra='satellite', re='satellite', cj='no', rr='uav')
DSS_LC = {'GEO_SAMPLER': 'dss', 'LC': '1'}
RUNS = [  # (name, overrides of BASE, script, environment), in priority order
    ('global', {}, 'train.py', {}),                                        # baseline
    ('global_dss_lc', {}, 'train_geo.py', DSS_LC),                         # O4 + CT2
    ('vitb_global', {'backbone': 'ViTB-224'}, 'train.py', {}),             # O2
    ('global_dss', {}, 'train_geo.py', {'GEO_SAMPLER': 'dss', 'LC': '0'}),  # O4
    ('global_ct2', {}, 'train_geo.py', {'GEO_SAMPLER': 'geo', 'LC': '1'}),  # CT2 as in the earlier notebook
    ('irp', {'head': 'IRP', 'block': 2, 'lr': 0.01}, 'train.py', {}),                           # CT1
    ('dino_global', {'backbone': 'DINOv2S-224'}, 'train.py', {}),          # O3
    ('global_seed777', {}, 'train.py', {'SEED': '777'}),                   # noise of the baseline
    ('vitb_global_dss_lc', {'backbone': 'ViTB-224'}, 'train_geo.py', DSS_LC),  # O2 + O4 + CT2
]
os.makedirs(CK, exist_ok=True)
os.makedirs(LOGS, exist_ok=True)
GPUS = queue.Queue()
for g in range(torch.cuda.device_count()):
    GPUS.put(g)
UNIT = []  # minutes per ViT-S-equivalent run, measured on the runs that finished


def job(run):
    name, over, script, extra_env = run
    a, ck, log_path = {**BASE, **over}, f'{CK}/{name}', f'{LOGS}/{name}.log'
    if os.path.exists(f'{ck}/SDM@K(1,100).json'):
        return name, 'done earlier'
    g = GPUS.get()
    try:
        est = (np.median(UNIT) if UNIT else 120) * COST[a['backbone']]  # 120 min = speed of the earlier full-resolution runs
        if time.time() - T0 + 60 * est > 3600 * DEADLINE_H:
            return name, f'skipped: ~{est:.0f} min would end after the {DEADLINE_H} h deadline'
        t = time.time()
        env = {**os.environ, 'CUDA_VISIBLE_DEVICES': str(g), **extra_env}
        args = [x for k, v in a.items() for x in (f'--{k}', str(v))]
        steps = [([sys.executable, script, '--name', name, '--data_dir', f'{DATA}/train', *args], REPO),
                 ([sys.executable, 'test.py', '--name', name, '--test_dir', f'{DATA}/test', '--gpu_ids', '0',
                   '--num_worker', str(a['num_worker'])], ck),
                 ([sys.executable, 'evaluate_gpu.py'], ck),
                 ([sys.executable, 'evaluateDistance.py', '--root_dir', DATA], ck)]
        with open(log_path, 'w') as log:
            for cmd, cwd in steps:
                subprocess.run(cmd, cwd=cwd, env=env, stdout=log, stderr=subprocess.STDOUT, check=True)
            with open(f'{ck}/infer.txt', 'w') as out:  # Params / MACs / InferTime; a failure here does not fail the run
                subprocess.run([sys.executable, 'get_inference_time.py'], cwd=f'{ck}/tool',
                               env={**env, 'PYTHONPATH': os.pathsep.join(['..', env.get('PYTHONPATH', '')])},
                               stdout=out, stderr=log)
        minutes = (time.time() - t) / 60
        UNIT.append(minutes / COST[a['backbone']])
        json.dump({'minutes': minutes, 'gpu': g}, open(f'{ck}/job.json', 'w'))
        return name, f'ok on GPU {g} in {minutes:.0f} min'
    except subprocess.CalledProcessError as e:
        tail = ''.join(open(log_path).readlines()[-15:])
        return name, f'FAILED in {os.path.basename(e.cmd[1])}, end of {log_path}:\n{tail}'
    finally:
        GPUS.put(g)

In [10]:
with ThreadPoolExecutor(max(1, torch.cuda.device_count())) as ex:
    for f in as_completed([ex.submit(job, r) for r in RUNS]):
        name, status = f.result()
        print(f'[{(time.time() - T0) / 3600:.2f} h] {name}: {status}', flush=True)

[2.09 h] global: ok on GPU 0 in 125 min
[2.31 h] global_dss_lc: ok on GPU 1 in 138 min
[4.37 h] vitb_global: ok on GPU 0 in 137 min
[4.71 h] global_dss: ok on GPU 1 in 144 min
[6.47 h] global_ct2: ok on GPU 0 in 126 min
[6.83 h] irp: ok on GPU 1 in 127 min
[8.51 h] dino_global: ok on GPU 0 in 122 min
[8.51 h] vitb_global_dss_lc: skipped: ~442 min would end after the 10.75 h deadline
[8.79 h] global_seed777: ok on GPU 1 in 118 min


## Kết quả theo giao thức của repo
Các số dưới đây lấy từ `results.txt` và `SDM@K(1,100).json` do script của repo ghi ra, với TTA lật ngang như `test.py`. Bảng có thêm vài dòng tham chiếu để so.

In [11]:
import pandas as pd

DONE = [r[0] for r in RUNS if os.path.exists(f'{CK}/{r[0]}/SDM@K(1,100).json')]


def repo_metrics(name):
    ck = f'{CK}/{name}'
    r = dict(re.findall(r'([\w@]+):([\d.]+)', open(f'{ck}/results.txt').read()))
    s = json.load(open(f'{ck}/SDM@K(1,100).json'))
    inf = dict(re.findall(r'(\w+)=([\d.]+[KMGs]?)', open(f'{ck}/infer.txt').read())) if os.path.exists(f'{ck}/infer.txt') else {}
    job_min = json.load(open(f'{ck}/job.json'))['minutes'] if os.path.exists(f'{ck}/job.json') else np.nan
    return {**{k: float(r[f'Recall@{k[2:]}']) for k in ('R@1', 'R@5', 'R@10')}, 'AP': float(r['AP']),
            **{f'SDM@{k}': 100 * s[str(k)] for k in (1, 3, 5, 10)},
            'Params': inf.get('Params'), 'Macs': inf.get('Macs'), 'InferTime': inf.get('InferTime'), 'run min': job_min}


T1 = pd.DataFrame({n: repo_metrics(n) for n in DONE}).T
for c in T1.columns[:8]:
    T1[c] = T1[c].astype(float)
if 'global' in DONE:
    T1.insert(1, 'ΔR@1 vs global', T1['R@1'] - T1.loc['global', 'R@1'])
REF = pd.DataFrame({
    'ref: Global paper setup, earlier full-resolution run': {'R@1': 79.45, 'R@5': 93.52, 'R@10': 97.00, 'AP': 69.10, 'SDM@1': 83.92},
    'ref: paper Global (Table VI)': {'R@1': 80.18, 'R@5': 93.99, 'SDM@1': 84.39, 'SDM@3': 82.51, 'SDM@5': 78.02, 'SDM@10': 66.46},
    'ref: paper ViT-B Global (Table V)': {'R@1': 87.82, 'R@5': 97.17},
    'ref: CEUSP, ConvNeXt-T 256 (arXiv 2502.11408)': {'R@1': 89.45, 'R@5': 96.05, 'AP': 79.62, 'SDM@1': 91.01, 'SDM@3': 89.42, 'SDM@5': 85.34},
}).T
display(pd.concat([T1, REF]).round(2))
if {'global', 'global_seed777'} <= set(DONE):
    print(f"seed noise of the baseline: |ΔR@1| = {abs(T1.loc['global', 'R@1'] - T1.loc['global_seed777', 'R@1']):.2f}, "
          f"|ΔSDM@1| = {abs(T1.loc['global', 'SDM@1'] - T1.loc['global_seed777', 'SDM@1']):.2f}")

,R@1,ΔR@1 vs global,R@5,R@10,AP,SDM@1,SDM@3,SDM@5,SDM@10,Params,Macs,InferTime,run min
global,77.48,0.00,93.35,96.74,67.75,82.02,80.65,76.96,66.01,22.926M,8.484G,40.3s,125.275131
global_dss_lc,80.35,2.87,95.02,97.85,68.90,84.80,83.16,78.92,67.76,22.926M,8.484G,41.4s,138.477906
vitb_global,87.99,10.51,97.73,99.27,79.30,90.34,89.35,85.74,73.17,87.160M,33.699G,69.5s,136.707871
global_dss,80.65,3.17,94.04,96.91,69.32,84.08,81.96,77.76,66.12,22.926M,8.484G,42.2s,144.176224
global_ct2,79.54,2.06,94.42,97.94,68.34,84.00,82.43,78.24,67.08,22.926M,8.484G,41.1s,126.208618
irp,83.35,5.87,95.32,97.94,72.83,86.19,84.95,80.71,68.23,26.030M,8.491G,46.8s,127.336194
dino_global,3.26,-74.22,9.57,15.19,2.47,6.17,5.80,5.68,5.48,22.857M,11.032G,44.1s,122.32549
global_seed777,78.21,0.73,92.71,96.31,67.12,82.64,80.71,76.59,65.36,22.926M,8.484G,27.3s,117.638493
"ref: Global paper setup, earlier full-resolution run",79.45,NaN,93.52,97.00,69.10,83.92,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ref: paper Global (Table VI),80.18,NaN,93.99,NaN,NaN,84.39,82.51,78.02,66.46,NaN,NaN,NaN,NaN


seed noise of the baseline: |ΔR@1| = 0.73, |ΔSDM@1| = 0.62


## O5: test-time (không train lại)
Ảnh test được giải mã một lần vào RAM (uint8, transform của `test.py`). Mỗi model chạy một lượt qua đủ 8 view D4, lưu cả đặc trưng 2 view lẫn 8 view.

| biến thể | query | gallery |
|---|---|---|
| `flip (test.py)` | gốc + lật | gốc + lật, giống repo. Dùng để kiểm tra rằng R@1 tính lại khớp `results.txt` |
| `D4 query` | 8 view | 2 view |
| `D4 both` | 8 view | 8 view |
| `D4 both + DBA years` | 8 view | mỗi ảnh vệ tinh được thay bằng trung bình với ảnh cùng ô, cùng tỉ lệ, năm còn lại (2020 ↔ 2022). **Chọn trước làm cấu hình cuối** |
| `D4 both + DBA place` | 8 view | trung bình cả 6 ảnh của một địa điểm (3 tỉ lệ × 2 năm) |

**Đọc kết quả:**
- DBA dùng thông tin mà bản đồ vệ tinh đã có sẵn: ảnh nào thuộc ô nào. Nó không dùng gì của query.
- Các ảnh trong một nhóm DBA có cùng đặc trưng, nên R@5/R@10/SDM@5 bị lệch theo cách đếm của repo. Với các dòng DBA chỉ báo R@1 và SDM@1.
- Chạy ở fp16, nên dòng `flip (test.py)` có thể lệch vài phần trăm của 1 điểm so với `results.txt` (fp32).

In [12]:
import yaml
import torch.nn.functional as F
from types import SimpleNamespace
from torchvision import datasets as tvd, transforms
from models.taskflow import make_model
from geo_utils import load_gps, class_xy, recall, sdm, areas, error_types

tf = transforms.Compose([transforms.Resize((224, 224), interpolation=3), transforms.PILToTensor()])  # test.py, as uint8
gal = tvd.ImageFolder(f'{DATA}/test/gallery_satellite', tf)
qry = tvd.ImageFolder(f'{DATA}/test/query_drone', tf)
load_all = lambda ds: torch.cat([x for x, _ in torch.utils.data.DataLoader(ds, 128, num_workers=NW)])
XG, XQ = load_all(gal), load_all(qry)
names = np.array(gal.classes)
g_lab = np.array(gal.targets)                                              # gallery image -> place index
q_lab = np.array([gal.class_to_idx[qry.classes[t]] for t in qry.targets])  # query image -> place index
gps = load_gps(DATA)
xy = class_xy(names, gps)                                                  # metres
ll = np.array([gps[n] for n in names])                                     # (lon, lat), for SDM
is_test = np.isin(np.arange(len(names)), q_lab)
area = areas(xy)
scale = [os.path.basename(p).split('.')[0].replace('_old', '') for p, _ in gal.imgs]  # H80_old.tif -> H80
year_grp = np.unique([f'{l}_{s}' for l, s in zip(g_lab, scale)], return_inverse=True)[1].ravel()
print(f'{len(names)} places, {len(g_lab)} gallery / {len(q_lab)} query images, {area.max() + 1} areas '
      f'({len(np.unique(area[is_test]))} with queries; paper: 14 universities, 4 for test); '
      f'DBA-year groups: {np.bincount(np.bincount(year_grp))[1:].tolist()} groups of size 1, 2, ...')

MEAN = torch.tensor([0.485, 0.456, 0.406], device='cuda').view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device='cuda').view(1, 3, 1, 1)
D4 = [(k, fl) for k in range(4) for fl in (False, True)]  # D4[:2] = identity + horizontal flip = test.py's TTA
norm = lambda f: F.normalize(F.normalize(f, dim=1).flatten(1), dim=1).cpu()  # per part as test.py, then whole vector


def load_model(name):
    opt = SimpleNamespace(**yaml.safe_load(open(f'{CK}/{name}/opts.yaml')))
    model = make_model(opt)
    model.load_state_dict(torch.load(f'{CK}/{name}/net_119.pth', map_location='cpu'))
    return model.cuda().eval()


@torch.no_grad()
def extract(model, X, bs=128):
    """(2-view, 8-view) features of X in one pass: views are summed, then normalised, as in test.py."""
    f2, f8 = [], []
    for i in range(0, len(X), bs):
        x = (X[i:i + bs].cuda().float() / 255 - MEAN) / STD
        acc = 0
        for j, (k, fl) in enumerate(D4):
            with torch.autocast('cuda', torch.float16):
                acc = acc + model(torch.rot90(x.flip(3) if fl else x, k, (2, 3)), None)[0][1].float()
            if j == 1:
                f2.append(norm(acc))
        f8.append(norm(acc))
    return torch.cat(f2).numpy(), torch.cat(f8).numpy()


@torch.no_grad()
def rank(qf, gf, k=100):
    g = torch.from_numpy(gf).cuda()
    return torch.cat([(torch.from_numpy(qf[i:i + 512]).cuda() @ g.T).topk(k).indices.cpu()
                      for i in range(0, len(qf), 512)]).numpy()


def group_mean(f, grp):
    m = np.zeros((grp.max() + 1, f.shape[1]), np.float32)
    np.add.at(m, grp, f)
    return m / np.linalg.norm(m, axis=1, keepdims=True)


def metrics(top, tied=False):
    out = {**{f'R@{k}': recall(top, q_lab, g_lab, k) for k in (1, 5, 10)},
           **{f'SDM@{k}': sdm(top, ll[q_lab], ll[g_lab], k) for k in (1, 5)}}
    return {k: (np.nan if tied and k in ('R@5', 'R@10', 'SDM@5') else v) for k, v in out.items()}

3033 places, 18198 gallery / 2331 query images, 13 areas (4 with queries; paper: 14 universities, 4 for test); DBA-year groups: [0, 9099] groups of size 1, 2, ...


In [13]:
FINAL = 'D4 both + DBA years'
TT, CT3IN = {}, {}
for n in DONE:
    model = load_model(n)
    (q2, q8), (g2, g8) = extract(model, XQ), extract(model, XG)
    del model
    torch.cuda.empty_cache()
    variants = {'flip (test.py)': (q2, g2), 'D4 query': (q8, g2), 'D4 both': (q8, g8),
                FINAL: (q8, group_mean(g8, year_grp)[year_grp]), 'D4 both + DBA place': (q8, group_mean(g8, g_lab)[g_lab])}
    for v, (q, g) in variants.items():
        top = rank(q, g)
        TT[n, v] = metrics(top, tied='DBA' in v)
        if v in ('flip (test.py)', FINAL):  # CT3 needs the top-1 place, place prototypes and query features
            CT3IN[n, v] = g_lab[top[:, 0]], group_mean(g, g_lab), q
    gap = TT[n, 'flip (test.py)']['R@1'] - T1.loc[n, 'R@1']
    assert abs(gap) < 1.0, f'{n}: recomputed R@1 is {gap:+.2f} off results.txt, the extraction differs from test.py'

T2 = pd.DataFrame(TT).T.rename_axis(['run', 'test-time'])
display(T2.round(2))
display(T2['R@1'].unstack().reindex(DONE).round(2).style.set_caption('R@1 by run (rows) and test-time variant (columns)'))
display(T2['SDM@1'].unstack().reindex(DONE).round(2).style.set_caption('SDM@1'))

R@1    R@5   R@10  SDM@1  SDM@5
run            test-time                                             
global         flip (test.py)       77.48  93.35  96.74  82.02  76.96
               D4 query             78.16  93.91  97.47  83.38  77.83
               D4 both              78.85  94.47  97.81  83.79  78.99
               D4 both + DBA years  83.31    NaN    NaN  87.17    NaN
               D4 both + DBA place  83.53    NaN    NaN  87.31    NaN
global_dss_lc  flip (test.py)       80.31  95.07  97.85  84.76  78.93
               D4 query             79.02  94.81  98.41  84.24  78.34
               D4 both              80.57  95.32  98.63  85.31  79.73
               D4 both + DBA years  85.11    NaN    NaN  88.92    NaN
               D4 both + DBA place  85.41    NaN    NaN  89.08    NaN
vitb_global    flip (test.py)       87.99  97.73  99.27  90.34  85.74
               D4 query             85.89  97.47  98.97  88.99  84.48
               D4 both              87.30  97.64  98.97  90.28  85.36
               D4 both + DBA years  90.69    NaN    NaN  92.99    NaN
               D4 both + DBA place  90.78    NaN    NaN  93.02    NaN
global_dss     flip (test.py)       80.65  94.08  96.91  84.08  77.75
               D4 query             80.78  95.24  97.85  85.11  79.12
               D4 both              81.25  95.15  98.07  85.67  80.24
               D4 both + DBA years  84.60    NaN    NaN  88.25    NaN
               D4 both + DBA place  84.47    NaN    NaN  87.92    NaN
global_ct2     flip (test.py)       79.49  94.42  97.94  83.96  78.24
               D4 query             78.85  94.98  97.98  84.13  78.40
               D4 both              79.84  94.81  98.11  84.96  79.58
               D4 both + DBA years  84.17    NaN    NaN  88.03    NaN
               D4 both + DBA place  84.30    NaN    NaN  88.08    NaN
irp            flip (test.py)       83.44  95.37  97.94  86.26  80.71
               D4 query             81.42  95.07  97.85  85.39  80.19
               D4 both              82.88  95.20  98.07  86.71  81.44
               D4 both + DBA years  86.27    NaN    NaN  89.46    NaN
               D4 both + DBA place  86.36    NaN    NaN  89.49    NaN
dino_global    flip (test.py)        3.26   9.57  15.19   6.17   5.68
               D4 query              3.26   9.65  15.10   6.15   5.73
               D4 both               3.00  10.34  15.19   6.10   5.98
               D4 both + DBA years   2.79    NaN    NaN   4.87    NaN
               D4 both + DBA place   2.62    NaN    NaN   4.64    NaN
global_seed777 flip (test.py)       78.16  92.71  96.31  82.58  76.59
               D4 query             77.43  93.35  97.21  82.69  77.29
               D4 both              78.81  93.69  97.34  83.73  78.63
               D4 both + DBA years  82.15    NaN    NaN  86.33    NaN
               D4 both + DBA place  82.54    NaN    NaN  86.79    NaN

test-time,D4 both,D4 both + DBA place,D4 both + DBA years,D4 query,flip (test.py)
run,,,,,
global,78.850000,83.530000,83.310000,78.160000,77.480000
global_dss_lc,80.570000,85.410000,85.110000,79.020000,80.310000
vitb_global,87.300000,90.780000,90.690000,85.890000,87.990000
global_dss,81.250000,84.470000,84.600000,80.780000,80.650000
global_ct2,79.840000,84.300000,84.170000,78.850000,79.490000
irp,82.880000,86.360000,86.270000,81.420000,83.440000
dino_global,3.000000,2.620000,2.790000,3.260000,3.260000
global_seed777,78.810000,82.540000,82.150000,77.430000,78.160000


test-time,D4 both,D4 both + DBA place,D4 both + DBA years,D4 query,flip (test.py)
run,,,,,
global,83.790000,87.310000,87.170000,83.380000,82.020000
global_dss_lc,85.310000,89.080000,88.920000,84.240000,84.760000
vitb_global,90.280000,93.020000,92.990000,88.990000,90.340000
global_dss,85.670000,87.920000,88.250000,85.110000,84.080000
global_ct2,84.960000,88.080000,88.030000,84.130000,83.960000
irp,86.710000,89.490000,89.460000,85.390000,86.260000
dino_global,6.100000,4.640000,4.870000,6.150000,6.170000
global_seed777,83.730000,86.790000,86.330000,82.690000,82.580000


## CT3: chẩn đoán lỗi
Mỗi query được xếp vào một loại theo dự đoán top-1:

| loại | dự đoán top-1 |
|---|---|
| C0 | đúng |
| C1 | sai ô nhưng cách ≤ 30 m, tức một bước lưới (vẫn dùng được để định vị) |
| C2 | cùng khu vực nhưng cách > 30 m (nhầm trong cùng campus) |
| C3 | một khu test khác (nhầm giữa các campus) |
| C4 | một địa điểm train. Campus train và test tách rời, nên đây cũng là nhầm giữa các campus |

**Nguyên nhân của lỗi xa (C2–C4):**
- *Cặp mơ hồ*: d_F(ô thật, ô dự đoán) < τ, với τ = trung vị khoảng cách từ query tới ô của chính nó. Khi đó biểu diễn không tách được hai địa điểm.
- *Lệch miền*: ngược lại, tức hai địa điểm vốn tách được nhưng đặc trưng của query bị lệch.

Ma trận chuyển cho biết mỗi run sửa đúng, hoặc làm sai thêm, những query nào so với `global` (cùng cấu hình test-time cuối).

In [14]:
CATS = ['C0', 'C1', 'C2', 'C3', 'C4']
summary, cats = {}, {}
for (n, v), (pred, P, qf) in CT3IN.items():
    cat, err = error_types(pred, q_lab, xy, area, is_test)
    far = np.isin(cat, ['C2', 'C3', 'C4'])
    tau = np.median(1 - (qf * P[q_lab]).sum(1))
    ambiguous = far & (1 - (P[q_lab] * P[pred]).sum(1) < tau)
    cats[n, v] = cat
    summary[n, v] = {**(pd.Series(cat).value_counts(normalize=True) * 100).reindex(CATS, fill_value=0).to_dict(),
                     'far errors from ambiguous pairs %': 100 * ambiguous.sum() / max(far.sum(), 1),
                     'median error m (wrong top-1)': np.median(err[cat != 'C0']) if (cat != 'C0').any() else 0.0}
display(pd.DataFrame(summary).T.rename_axis(['run', 'test-time']).round(2))
if 'global' in DONE:
    for n in DONE[1:]:
        display(pd.crosstab(pd.Series(cats['global', FINAL], name='global'), pd.Series(cats[n, FINAL], name=n)))

C0     C1     C2     C3     C4  \
run            test-time                                                
global         flip (test.py)       77.48  11.33   6.86   1.89   2.45   
               D4 both + DBA years  83.31   9.65   4.63   1.03   1.37   
global_dss_lc  flip (test.py)       80.31  11.15   6.09   0.82   1.63   
               D4 both + DBA years  85.11   9.82   3.69   0.39   0.99   
vitb_global    flip (test.py)       87.99   5.83   4.12   0.77   1.29   
               D4 both + DBA years  90.69   5.75   2.45   0.39   0.73   
global_dss     flip (test.py)       80.65   8.37   6.05   1.46   3.47   
               D4 both + DBA years  84.60   9.27   3.82   0.73   1.59   
global_ct2     flip (test.py)       79.49  11.33   5.75   0.64   2.79   
               D4 both + DBA years  84.17   9.82   3.69   0.47   1.84   
irp            flip (test.py)       83.44   6.65   6.39   1.03   2.49   
               D4 both + DBA years  86.27   8.02   3.56   0.69   1.46   
dino_global    flip (test.py)        3.26   6.99  12.91  13.21  63.62   
               D4 both + DBA years   2.79   5.19   8.88  12.10  71.04   
global_seed777 flip (test.py)       78.16  11.11   6.65   1.37   2.70   
               D4 both + DBA years  82.15  10.60   4.42   0.51   2.32   

                                    far errors from ambiguous pairs %  \
run            test-time                                                
global         flip (test.py)                                   32.95   
               D4 both + DBA years                              30.49   
global_dss_lc  flip (test.py)                                   39.70   
               D4 both + DBA years                              38.14   
vitb_global    flip (test.py)                                   39.58   
               D4 both + DBA years                              43.37   
global_dss     flip (test.py)                                   26.95   
               D4 both + DBA years                              32.87   
global_ct2     flip (test.py)                                   37.85   
               D4 both + DBA years                              35.71   
irp            flip (test.py)                                   29.00   
               D4 both + DBA years                              31.58   
dino_global    flip (test.py)                                   47.13   
               D4 both + DBA years                              48.76   
global_seed777 flip (test.py)                                   34.40   
               D4 both + DBA years                              30.77   

                                    median error m (wrong top-1)  
run            test-time                                          
global         flip (test.py)                              29.48  
               D4 both + DBA years                         21.36  
global_dss_lc  flip (test.py)                              25.11  
               D4 both + DBA years                         20.66  
vitb_global    flip (test.py)                              39.58  
               D4 both + DBA years                         20.59  
global_dss     flip (test.py)                              44.05  
               D4 both + DBA years                         20.73  
global_ct2     flip (test.py)                              27.32  
               D4 both + DBA years                         20.71  
irp            flip (test.py)                              42.78  
               D4 both + DBA years                         20.86  
dino_global    flip (test.py)                            1560.81  
               D4 both + DBA years                       1637.97  
global_seed777 flip (test.py)                              29.34  
               D4 both + DBA years                         21.10

global_dss_lc,C0,C1,C2,C3,C4
global,,,,,
C0,1872,42,24,0,4
C1,53,169,1,0,2
C2,44,7,53,0,4
C3,4,5,4,8,3
C4,11,6,4,1,10


vitb_global,C0,C1,C2,C3,C4
global,,,,,
C0,1874,40,23,1,4
C1,131,85,3,0,6
C2,70,6,29,1,2
C3,15,2,0,7,0
C4,24,1,2,0,5


global_dss,C0,C1,C2,C3,C4
global,,,,,
C0,1887,28,17,3,7
C1,42,176,1,0,6
C2,31,6,68,1,2
C3,4,3,0,12,5
C4,8,3,3,1,17


global_ct2,C0,C1,C2,C3,C4
global,,,,,
C0,1845,61,19,3,14
C1,61,157,3,0,4
C2,37,6,61,0,4
C3,6,3,2,8,5
C4,13,2,1,0,16


irp,C0,C1,C2,C3,C4
global,,,,,
C0,1845,54,25,2,16
C1,86,125,7,0,7
C2,55,5,45,1,2
C3,9,1,2,11,1
C4,16,2,4,2,8


dino_global,C0,C1,C2,C3,C4
global,,,,,
C0,62,112,170,231,1367
C1,1,7,24,31,162
C2,2,2,9,12,83
C3,0,0,1,2,21
C4,0,0,3,6,23


global_seed777,C0,C1,C2,C3,C4
global,,,,,
C0,1824,68,29,5,16
C1,48,162,6,0,9
C2,34,10,57,1,6
C3,5,3,6,5,5
C4,4,4,5,1,18


## Tài liệu và dataset (subagent đọc nguồn gốc, kiểm tra ngày 2026-10-18)
**Kết quả công bố trên DenseUAV (drone → vệ tinh, đủ 3 độ cao):**

| R@1 | SDM@1 | Phương pháp | Backbone | Ghi chú |
|---|---|---|---|---|
| 97.17 | 97.68 | DINO-GFSA (arXiv 2606.00784) | DINOv3 ViT-L + LoRA | code chưa công bố |
| 95.22 | 95.23 | MFAF (arXiv 2509.12673) | EVA02-L, 448 px | không có code |
| 91.85 | 93.59 | SkyPart (arXiv 2605.11654) | 27M tham số | không có code |
| 89.45 | 91.01 | CEUSP (arXiv 2502.11408) | ConvNeXt-T, 256 px | nguồn của O4 |
| 86.27 | 88.87 | Yang et al., RA-L 2025 | DINOv2-B | nguồn của O3 |
| 83.01 | 86.50 | DenseUAV baseline (TIP 2024) | ViT-S | |
| 49.38 | 61.72 | Sample4Geo (do CEUSP chạy lại) | ConvNeXt-B | InfoNCE + negative theo GPS chạy kém trên lưới dày |

Bài học từ Sample4Geo và nhánh chỉ-GPS của CEUSP: các điểm cách 20 m gần như trùng ảnh, nên ép chúng làm negative khó sẽ làm hỏng mô hình. Đó là lý do O4 trộn lớp dễ nhầm theo đặc trưng, và CT2 dùng loss thứ tự mềm thay vì đẩy láng giềng ra xa.

**Dataset khác:**

| dataset | góc nhìn / độ cao | GPS | có trên Kaggle? | dùng cho DenseUAV? |
|---|---|---|---|---|
| University-1652 | drone tổng hợp quanh tòa nhà | có (file riêng) | bản đầy đủ phải xin qua form; `hanahariri/university-1652` có thể là tập con | pretrain được, nhưng chưa có bằng chứng giúp DenseUAV |
| SUES-200 | 150–300 m | không | không | không có GPS → không dùng được cho CT2 |
| GTA-UAV | tổng hợp, xiên, 80–650 m | có (IoU) | không (HF 12.8 GB) | khác góc nhìn, quá nặng cho 12 h |
| UAV-VisLoc | 400–840 m, có góc heading | có | `hailong1610/uav-visloc-dataset` (17.7 GB) | độ cao khác xa; có thể dùng để thử cross-dataset |
| VPAIR | > 300 m, nhìn xuống | có | `muneeburrehman98/deepsim-vpair-preprocessed` | như trên |

**Kết luận:** trong một phiên 12 h nên ở lại DenseUAV. Đây là bộ duy nhất vừa nhìn thẳng xuống, vừa bay thấp, vừa là ảnh thật có GPS. Về building footprint cho CT1: Google Open Buildings không phủ Trung Quốc, Microsoft GlobalML chưa rõ mức phủ, và DenseUAV không kèm mask. Muốn dùng polygon thật thì phải chạy một mô hình segmentation trên ảnh vệ tinh.

In [15]:
shutil.rmtree(DATA, ignore_errors=True)  # the drone cache (~1 GB); symlinks inside are removed, not followed
print('kept in /kaggle/working:', sorted(os.listdir(CK)), '| logs in', LOGS)

kept in /kaggle/working: ['dino_global', 'global', 'global_ct2', 'global_dss', 'global_dss_lc', 'global_seed777', 'irp', 'vitb_global'] | logs in /kaggle/working/logs
